In [3]:
# Import libraries
from google.cloud import bigquery
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

PROJECT_ID = 'musa5090s26-team5'
client = bigquery.Client(project=PROJECT_ID)

# Load training data directly from core.opa_properties
query = """
SELECT
    parcel_number,
    sale_price,
    sale_date,
    total_livable_area,
    total_area,
    number_of_bedrooms,
    number_of_bathrooms,
    number_stories,
    garage_spaces,
    fireplaces,
    exterior_condition,
    interior_condition,
    quality_grade,
    year_built,
    zip_code,
    zoning,
    category_code,
    building_code_new
FROM `musa5090s26-team5.core.opa_properties`
WHERE category_code = '1'
    AND sale_price > 1
    AND sale_date >= '2015-01-01'
    AND sale_price IS NOT NULL
    AND total_livable_area IS NOT NULL
    AND total_livable_area > 0
"""

df = client.query(query).to_dataframe(create_bqstorage_client=False)
print(f"Rows: {len(df):,}")
print(df.dtypes)
df.head()

C:\ProgramData\anaconda3\envs\geospatial\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.15) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\ProgramData\anaconda3\envs\geospatial\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.15) which Google will stop supporting in new releases of google.cloud.bigquery_storage_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.bigquery_storage_v1 past that date.
  warnings.warn(message, FutureWarning)


Rows: 171,304
parcel_number           object
sale_price             float64
sale_date               object
total_livable_area     float64
total_area             float64
number_of_bedrooms     float64
number_of_bathrooms    float64
number_stories         float64
garage_spaces          float64
fireplaces             float64
exterior_condition      object
interior_condition      object
quality_grade           object
year_built              object
zip_code                object
zoning                  object
category_code           object
building_code_new       object
dtype: object


,parcel_number,sale_price,sale_date,total_livable_area,total_area,number_of_bedrooms,number_of_bathrooms,number_stories,garage_spaces,fireplaces,exterior_condition,interior_condition,quality_grade,year_built,zip_code,zoning,category_code,building_code_new
0,021165600,865000.0,2022-10-10T04:00:00Z,2673.0,1168.0,4.0,3.0,3.0,1.0,0.0,2,2,None,1920,19147,RSA5,1,22
1,888001611,410000.0,2025-09-18T04:00:00Z,605.0,0.0,1.0,1.0,1.0,NaN,NaN,3,2,None,1900,19103,RMX3,1,41
2,022000006,860000.0,2020-09-18T04:00:00Z,1960.0,1314.0,NaN,NaN,NaN,NaN,NaN,4,4,C,1989,19147,RSA5,1,820
3,182045505,435000.0,2018-10-04T04:00:00Z,1283.0,840.0,3.0,3.0,2.0,0.0,0.0,3,3,B,2017,19122,RSA5,1,25
4,181320210,591000.0,2022-03-09T05:00:00Z,2639.0,1301.0,3.0,3.0,3.0,1.0,0.0,3,3,C,2017,19125,RSA5,1,25


In [4]:
# Feature engineering and model training
from sklearn.preprocessing import LabelEncoder

# Select numeric + categorical features
features = [
    'total_livable_area', 'total_area', 'number_of_bedrooms',
    'number_of_bathrooms', 'number_stories', 'garage_spaces',
    'fireplaces', 'exterior_condition', 'interior_condition',
    'quality_grade', 'year_built', 'zip_code'
]

df_model = df[features + ['sale_price']].copy()

# Fill nulls
for col in features:
    if df_model[col].dtype == 'object':
        df_model[col] = df_model[col].fillna('Unknown')
        le = LabelEncoder()
        df_model[col] = le.fit_transform(df_model[col].astype(str))
    else:
        df_model[col] = df_model[col].fillna(df_model[col].median())

# Remove extreme outliers in sale_price
q99 = df_model['sale_price'].quantile(0.99)
df_model = df_model[df_model['sale_price'] <= q99]

print(f"Training rows after filtering: {len(df_model):,}")

# Train/test split
X = df_model[features]
y = df_model['sale_price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
print("Training model...")
model = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MAE: ${mae:,.0f}")
print(f"R²: {r2:.4f}")

Training rows after filtering: 169,592
Training model...
MAE: $72,698
R²: 0.6862


In [5]:
import pickle
from datetime import datetime, timezone

# Save model locally
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Model saved.")

# Load all residential properties for prediction
query_all = """
SELECT
    parcel_number,
    total_livable_area,
    total_area,
    number_of_bedrooms,
    number_of_bathrooms,
    number_stories,
    garage_spaces,
    fireplaces,
    exterior_condition,
    interior_condition,
    quality_grade,
    year_built,
    zip_code,
    zoning,
    category_code,
    building_code_new
FROM `musa5090s26-team5.core.opa_properties`
WHERE category_code = '1'
    AND total_livable_area IS NOT NULL
    AND total_livable_area > 0
"""

df_all = client.query(query_all).to_dataframe(create_bqstorage_client=False)
print(f"Properties to predict: {len(df_all):,}")

# Apply same feature engineering
df_pred = df_all[features].copy()
for col in features:
    if df_pred[col].dtype == 'object':
        df_pred[col] = df_pred[col].fillna('Unknown')
        le = LabelEncoder()
        df_pred[col] = le.fit_transform(df_pred[col].astype(str))
    else:
        df_pred[col] = df_pred[col].fillna(df_pred[col].median())

# Predict
predicted_values = model.predict(df_pred)
predicted_at = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')

# Build results dataframe
df_results = pd.DataFrame({
    'property_id': df_all['parcel_number'],
    'predicted_value': predicted_values.round(2),
    'predicted_at': predicted_at
})

print(df_results.head())
print(f"Total predictions: {len(df_results):,}")

Model saved.
Properties to predict: 462,669
  property_id  predicted_value         predicted_at
0   021165600        875598.06  2026-04-15 19:34:20
1   888001611        287246.39  2026-04-15 19:34:20
2   022000006        400292.83  2026-04-15 19:34:20
3   182045505        236945.05  2026-04-15 19:34:20
4   181320210        430334.39  2026-04-15 19:34:20
Total predictions: 462,669


In [6]:
from google.cloud import bigquery

# Write predictions to BigQuery
table_id = 'musa5090s26-team5.derived.current_assessments'

job_config = bigquery.LoadJobConfig(
    write_disposition='WRITE_TRUNCATE',
    schema=[
        bigquery.SchemaField('property_id', 'STRING'),
        bigquery.SchemaField('predicted_value', 'FLOAT64'),
        bigquery.SchemaField('predicted_at', 'STRING'),
    ]
)

job = client.load_table_from_dataframe(df_results, table_id, job_config=job_config)
job.result()

print(f"Loaded {len(df_results):,} rows to {table_id}")

Loaded 462,669 rows to musa5090s26-team5.derived.current_assessments
